<a href="https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring (Lane 2)

ML task type: this is primarily a RANKING / SCORING problem, not a hard
classification problem. The goal isn't to classify each page as "refresh" or
"don't refresh" in isolation — it's to produce an ordered list of pages so a
reviewer with limited capacity works through the most promising candidates first.

Under the hood, this is built on top of a binary classification model (declining
vs not declining) whose predicted probability is used as the ranking score —
this mirrors exactly what the starter pipeline does (random forest probability
-> ranked queue -> Precision@K evaluation).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Starter proxy target: is_declining_label = (trend_direction == "down")

This is a PROXY, not the real outcome I ultimately care about. It's calculated
from the current window, not a validated future event. A stronger target for
later weeks would be a future-window label, e.g.:

  features from prior 90 days -> decline over the next 30 days

I'll start with the starter proxy for this task framing exercise, and note this
limitation explicitly (per the lane guide's own warning about proxy labels).

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeeljames/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check clone worked"
print("Repo cloned and ready.")

Working dir: /content/flyrank-ml-internship
Repo cloned and ready.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K (specifically Precision@50, matching the starter
pipeline and the reviewer's realistic weekly capacity).

Why not plain accuracy: accuracy treats all pages equally, but a reviewer only
ever looks at their top N candidates. Precision@50 asks the metric that actually
matters for this decision: "of the top 50 pages the system flags, how many are
genuinely declining?" This directly measures whether a reviewer's limited time
is being spent well.

Starter pipeline benchmark for reference:
- baseline hand rule: Precision@50 = 0.240
- random forest:      Precision@50 = 0.740

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page (one content_id), for a single
client. This matches the grain of the starter dataset and the warehouse's
dim_content table.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the unit of analysis as an actual dataframe
unit_of_analysis = df[[
    "content_id", "client_id", "trend_direction", "is_declining_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr"
]].head(10)

print("Shape:", df.shape, "-> one row per content page")
unit_of_analysis

Shape: (30000, 45) -> one row per content page


,content_id,client_id,trend_direction,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr
0,content_304f48230142,client_f369cb89fc,down,1,3803,20,10.6,0.76
1,content_a1fb4e703a9e,client_4e07408562,down,1,15320,25,20.3,0.05
2,content_9aa793d4d895,client_7f2253d7e2,down,1,12581,20,36.5,0.09
3,content_331d6c4de07b,client_19581e27de,stable,0,11751,22,6.2,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,down,1,19140,14,44.0,0.13
5,content_d4084a4bc775,client_f369cb89fc,down,1,3970,20,8.5,0.03
6,content_9a34b442b552,client_8722616204,down,1,20,20,7.0,0.00
7,content_a63219c6e95a,client_19581e27de,stable,0,1724,22,21.2,0.06
8,content_5e6c160719bc,client_6208ef0f77,down,1,32574,20,46.0,0.09
9,content_c27558df2b0c,client_19581e27de,down,1,1240,104,4.9,0.16


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "stale AND visible") is a single hard threshold — it can only
say yes/no on one or two conditions at a time, and my own Week 1 data check
showed that rule alone matches only 0.1% of pages, far too narrow to be useful
on its own.

The starter pipeline's own comparison shows this concretely: the hand-written
baseline rule reaches Precision@50 = 0.240, while a random forest reaches
Precision@50 = 0.740 — roughly 3x better. This suggests the real signal is a
combination of many weak, correlated features (impressions, position, freshness,
CTR, engagement) that no single hand-written rule can capture well, but a model
can learn from examples. That's exactly the kind of problem ML is suited for:
many weak signals, no single obvious threshold, and enough historical examples
to learn from.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.